# Selección de variables predictoras — Lending Club

Este notebook estudia si podemos reducir las **65 predictoras originales** usadas por el baseline. Sigue la partición temporal: entrenamiento antes de 2015, validación 2015–2016 y prueba desde 2017, que se mantiene reservada.

Comparamos regresión logística L1, importancia por permutación agrupada por variable original, eliminación recursiva agrupada y PCA. Las variables se seleccionan usando entrenamiento o un bloque interno de entrenamiento; validación se usa solo para comparar los conjuntos finales. PCA reduce dimensiones, pero no selecciona variables originales.

> **Resultados históricos:** este análisis usa el conjunto anterior de 65 predictoras y el umbral de 95%. La exploración final actual conserva 59 predictoras con menos del 10% de faltantes. Los resultados de selección de este notebook no representan el conjunto actualizado y deben recalcularse antes de usarlos.


In [1]:
from pathlib import Path
import sys, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from src.config import localizar_datos, ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA, UMBRAL_FALTANTES, UMBRAL_REVISION, SEMILLA
from src.data import inferir_tipos
from src.features import seleccionar_predictoras, preparar_matriz, construir_preprocesador

archivo = localizar_datos()
tipos_auditados = {'acc_now_delinq': 'Int8', 'acc_open_past_24mths': 'Int8', 'addr_state': 'category', 'all_util': 'Int16', 'annual_inc': 'float64', 'annual_inc_joint': 'float64', 'application_type': 'category', 'avg_cur_bal': 'Int32', 'bc_open_to_buy': 'Int32', 'bc_util': 'float64', 'chargeoff_within_12_mths': 'Int8', 'collection_recovery_fee': 'float64', 'collections_12_mths_ex_med': 'Int8', 'debt_settlement_flag': 'category', 'debt_settlement_flag_date': 'category', 'deferral_term': 'Int8', 'delinq_2yrs': 'Int8', 'delinq_amnt': 'Int32', 'desc': 'string', 'disbursement_method': 'category', 'dti': 'float64', 'dti_joint': 'float64', 'earliest_cr_line': 'string', 'emp_length': 'category', 'emp_title': 'string', 'fico_range_high': 'Int16', 'fico_range_low': 'Int16', 'funded_amnt': 'Int32', 'funded_amnt_inv': 'float64', 'grade': 'category', 'hardship_amount': 'float64', 'hardship_dpd': 'Int8', 'hardship_end_date': 'category', 'hardship_flag': 'category', 'hardship_last_payment_amount': 'float64', 'hardship_length': 'Int8', 'hardship_loan_status': 'category', 'hardship_payoff_balance_amount': 'float64', 'hardship_reason': 'category', 'hardship_start_date': 'category', 'hardship_status': 'category', 'hardship_type': 'category', 'home_ownership': 'category', 'id': 'string', 'il_util': 'Int16', 'initial_list_status': 'category', 'inq_fi': 'Int8', 'inq_last_12m': 'Int8', 'inq_last_6mths': 'Int8', 'installment': 'float64', 'int_rate': 'float64', 'issue_d': 'string', 'last_credit_pull_d': 'category', 'last_fico_range_high': 'Int16', 'last_fico_range_low': 'Int16', 'last_pymnt_amnt': 'float64', 'last_pymnt_d': 'category', 'loan_amnt': 'Int32', 'loan_status': 'category', 'max_bal_bc': 'Int32', 'member_id': 'string', 'mo_sin_old_il_acct': 'Int16', 'mo_sin_old_rev_tl_op': 'Int16', 'mo_sin_rcnt_rev_tl_op': 'Int16', 'mo_sin_rcnt_tl': 'Int16', 'mort_acc': 'Int8', 'mths_since_last_delinq': 'Int16', 'mths_since_last_major_derog': 'Int16', 'mths_since_last_record': 'Int16', 'mths_since_rcnt_il': 'Int16', 'mths_since_recent_bc': 'Int16', 'mths_since_recent_bc_dlq': 'Int16', 'mths_since_recent_inq': 'Int8', 'mths_since_recent_revol_delinq': 'Int16', 'next_pymnt_d': 'category', 'num_accts_ever_120_pd': 'Int8', 'num_actv_bc_tl': 'Int8', 'num_actv_rev_tl': 'Int8', 'num_bc_sats': 'Int8', 'num_bc_tl': 'Int8', 'num_il_tl': 'Int16', 'num_op_rev_tl': 'Int8', 'num_rev_accts': 'Int16', 'num_rev_tl_bal_gt_0': 'Int8', 'num_sats': 'Int8', 'num_tl_120dpd_2m': 'Int8', 'num_tl_30dpd': 'Int8', 'num_tl_90g_dpd_24m': 'Int8', 'num_tl_op_past_12m': 'Int8', 'open_acc': 'Int8', 'open_acc_6m': 'Int8', 'open_act_il': 'Int8', 'open_il_12m': 'Int8', 'open_il_24m': 'Int8', 'open_rv_12m': 'Int8', 'open_rv_24m': 'Int8', 'orig_projected_additional_accrued_interest': 'float64', 'out_prncp': 'float64', 'out_prncp_inv': 'float64', 'payment_plan_start_date': 'category', 'pct_tl_nvr_dlq': 'float64', 'percent_bc_gt_75': 'float64', 'policy_code': 'category', 'pub_rec': 'Int8', 'pub_rec_bankruptcies': 'Int8', 'purpose': 'category', 'pymnt_plan': 'category', 'recoveries': 'float64', 'revol_bal': 'Int32', 'revol_bal_joint': 'Int32', 'revol_util': 'float64', 'sec_app_chargeoff_within_12_mths': 'Int8', 'sec_app_collections_12_mths_ex_med': 'Int8', 'sec_app_earliest_cr_line': 'string', 'sec_app_fico_range_high': 'Int16', 'sec_app_fico_range_low': 'Int16', 'sec_app_inq_last_6mths': 'Int8', 'sec_app_mort_acc': 'Int8', 'sec_app_mths_since_last_major_derog': 'Int16', 'sec_app_num_rev_accts': 'Int8', 'sec_app_open_acc': 'Int8', 'sec_app_open_act_il': 'Int8', 'sec_app_revol_util': 'float64', 'settlement_amount': 'float64', 'settlement_date': 'category', 'settlement_percentage': 'float64', 'settlement_status': 'category', 'settlement_term': 'Int16', 'sub_grade': 'category', 'tax_liens': 'Int8', 'term': 'category', 'title': 'string', 'tot_coll_amt': 'Int32', 'tot_cur_bal': 'Int32', 'tot_hi_cred_lim': 'Int32', 'total_acc': 'Int16', 'total_bal_ex_mort': 'Int32', 'total_bal_il': 'Int32', 'total_bc_limit': 'Int32', 'total_cu_tl': 'Int8', 'total_il_high_credit_limit': 'Int32', 'total_pymnt': 'float64', 'total_pymnt_inv': 'float64', 'total_rec_int': 'float64', 'total_rec_late_fee': 'float64', 'total_rec_prncp': 'float64', 'total_rev_hi_lim': 'Int32', 'url': 'string', 'verification_status': 'category', 'verification_status_joint': 'category', 'zip_code': 'category'}
nombres_variables = {'acc_now_delinq': 'cuentas_en_mora_actual', 'acc_open_past_24mths': 'cuentas_abiertas_ultimos_24m', 'annual_inc': 'ingreso_anual', 'avg_cur_bal': 'saldo_promedio_cuentas', 'bc_open_to_buy': 'credito_disponible_tarjetas', 'bc_util': 'utilizacion_tarjetas_pct', 'chargeoff_within_12_mths': 'cuentas_castigadas_12m', 'collections_12_mths_ex_med': 'cobranzas_no_medicas_12m', 'delinq_2yrs': 'moras_ultimos_2_anios', 'delinq_amnt': 'monto_en_mora_actual', 'dti': 'ratio_deuda_ingreso', 'fico_range_high': 'fico_limite_superior', 'fico_range_low': 'fico_limite_inferior', 'inq_last_6mths': 'consultas_credito_6m', 'loan_amnt': 'monto_prestamo', 'mo_sin_old_il_acct': 'meses_desde_cuenta_plazos_mas_antigua', 'mo_sin_old_rev_tl_op': 'meses_desde_revolvente_mas_antigua', 'mo_sin_rcnt_rev_tl_op': 'meses_desde_revolvente_mas_reciente', 'mo_sin_rcnt_tl': 'meses_desde_cuenta_mas_reciente', 'mort_acc': 'cuentas_hipotecarias', 'mths_since_last_delinq': 'meses_desde_ultima_mora', 'mths_since_last_major_derog': 'meses_desde_ultima_mora_grave', 'mths_since_last_record': 'meses_desde_ultimo_registro_publico', 'mths_since_recent_bc': 'meses_desde_tarjeta_mas_reciente', 'mths_since_recent_bc_dlq': 'meses_desde_ultima_mora_tarjeta', 'mths_since_recent_inq': 'meses_desde_ultima_consulta_credito', 'mths_since_recent_revol_delinq': 'meses_desde_ultima_mora_revolvente', 'num_accts_ever_120_pd': 'cuentas_con_mora_120_dias', 'num_actv_bc_tl': 'tarjetas_bancarias_activas', 'num_actv_rev_tl': 'cuentas_revolventes_activas', 'num_bc_sats': 'tarjetas_bancarias_al_dia', 'num_bc_tl': 'total_cuentas_tarjeta', 'num_il_tl': 'total_cuentas_plazos', 'num_op_rev_tl': 'cuentas_revolventes_abiertas', 'num_rev_accts': 'total_cuentas_revolventes', 'num_rev_tl_bal_gt_0': 'revolventes_con_saldo', 'num_sats': 'cuentas_satisfactorias', 'num_tl_120dpd_2m': 'cuentas_mora_120_dias_ultimos_2m', 'num_tl_30dpd': 'cuentas_mora_30_dias_actual', 'num_tl_90g_dpd_24m': 'cuentas_mora_90_mas_dias_24m', 'num_tl_op_past_12m': 'cuentas_abiertas_12m', 'open_acc': 'lineas_credito_abiertas', 'pct_tl_nvr_dlq': 'pct_cuentas_sin_mora', 'percent_bc_gt_75': 'pct_tarjetas_utilizacion_mayor_75', 'pub_rec': 'registros_publicos_adversos', 'pub_rec_bankruptcies': 'quiebras_registradas', 'revol_bal': 'saldo_credito_revolvente', 'revol_util': 'utilizacion_credito_revolvente_pct', 'tax_liens': 'gravamenes_fiscales', 'term': 'plazo_meses', 'tot_coll_amt': 'monto_total_cobranzas', 'tot_cur_bal': 'saldo_total_actual', 'tot_hi_cred_lim': 'limite_credito_total', 'total_acc': 'total_lineas_credito', 'total_bal_ex_mort': 'saldo_total_sin_hipoteca', 'total_bc_limit': 'limite_total_tarjetas', 'total_il_high_credit_limit': 'limite_total_creditos_plazos', 'total_rev_hi_lim': 'limite_total_credito_revolvente', 'emp_length': 'antiguedad_laboral', 'home_ownership': 'tenencia_vivienda', 'verification_status': 'verificacion_ingresos', 'purpose': 'proposito_prestamo', 'addr_state': 'estado_residencia', 'application_type': 'tipo_solicitud', 'earliest_cr_line': 'fecha_primera_linea_credito'}
print("Archivo:", archivo)
print("Se evalúan las predictoras del escenario base, sin variables de precio Lending Club ni indicadores de faltantes.")

Archivo: C:\Users\josue\Downloads\accepted_2007_to_2018Q4.csv.gz
Se evalúan las predictoras del escenario base, sin variables de precio Lending Club ni indicadores de faltantes.


## 1. Cargar las columnas del escenario base y separar por tiempo

Leemos las columnas del mismo escenario usado por el baseline final. Filtramos préstamos con desenlace conocido y conservamos la separación temporal. El conjunto de prueba no participa en la selección ni en la comparación.

In [2]:
columnas_fuente = pd.read_csv(archivo, nrows=0).columns.tolist()
tipos_inferidos, _ = inferir_tipos(archivo)
tabla_columnas = pd.DataFrame(columns=columnas_fuente)
candidatas = seleccionar_predictoras(tabla_columnas, [], incluir_precio_lc=False)
columnas_lectura = [c for c in columnas_fuente if c in candidatas + ["loan_status", "issue_d"]]
dtype_lectura = {}
for columna in columnas_lectura:
    sugerido = tipos_auditados.get(columna, tipos_inferidos[columna])
    dtype_lectura[columna] = sugerido if sugerido not in ["category", "string"] else "string"

inicio = time.perf_counter()
df = pd.read_csv(archivo, usecols=columnas_lectura, dtype=dtype_lectura,
                 skipinitialspace=True, low_memory=False)
for columna in df.select_dtypes(include="string").columns:
    df[columna] = df[columna].astype("category")
for columna in ["issue_d", "earliest_cr_line", "sec_app_earliest_cr_line"]:
    if columna in df.columns:
        df[columna] = pd.to_datetime(df[columna].astype("string"), format="%b-%Y", errors="coerce")
if "term" in df.columns:
    df["term"] = pd.to_numeric(df["term"].astype("string").str.strip().str.replace(" months", "", regex=False), errors="coerce").astype("Int8")
columnas_vacias = df.columns[df.isna().all()].tolist()
if columnas_vacias:
    df = df.drop(columns=columnas_vacias)
    candidatas = [c for c in candidatas if c not in columnas_vacias]

estados = ["Fully Paid", "Charged Off"]
df = df.loc[df["loan_status"].isin(estados)].copy()
df["default"] = df["loan_status"].map({"Fully Paid": 0, "Charged Off": 1}).astype("int8")
df["fecha_prestamo"] = df["issue_d"]
df["anio"] = df["issue_d"].dt.year
assert df["anio"].notna().all()
train_df = df.loc[df.anio < ANIO_CORTE_VALIDACION].copy()
valid_df = df.loc[df.anio.between(ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA - 1)].copy()
prueba_filas = int((df.anio >= ANIO_CORTE_PRUEBA).sum())
print(f"Lectura: {len(df):,} préstamos elegibles, {df.shape[1]} columnas cargadas en {time.perf_counter()-inicio:.1f} s")
print(f"Entrenamiento: {len(train_df):,}; validación: {len(valid_df):,}; prueba reservada: {prueba_filas:,}")
print("Columnas completamente vacías retiradas:", len(columnas_vacias))

Lectura: 1,345,310 préstamos elegibles, 100 columnas cargadas en 92.4 s
Entrenamiento: 451,059; validación: 668,640; prueba reservada: 225,611
Columnas completamente vacías retiradas: 0


## 2. Reconstruir las 65 predictoras originales

Las variables con más de 95% de faltantes en entrenamiento se excluyen, siguiendo el notebook final. Las fechas del historial se convierten en antigüedad en meses. El preprocesador aprende medianas, categorías y escalas solo con entrenamiento.

In [3]:
faltantes_train_pct = train_df[candidatas].isna().mean() * 100
columnas_mas95 = faltantes_train_pct[faltantes_train_pct > UMBRAL_FALTANTES].index.tolist()
train_raw = preparar_matriz(train_df[candidatas + ["fecha_prestamo"]]).drop(columns="fecha_prestamo")
valid_raw = preparar_matriz(valid_df[candidatas + ["fecha_prestamo"]]).drop(columns="fecha_prestamo")
mapa_fecha = {
    "earliest_cr_line": "meses_desde_earliest_cr_line",
    "sec_app_earliest_cr_line": "meses_desde_sec_app_earliest_cr_line",
}
excluidas_preparadas = [mapa_fecha.get(c, c) for c in columnas_mas95]
columnas_modelo = [c for c in train_raw.columns if c not in excluidas_preparadas]
print("Predictoras originales antes de revisar faltantes:", len(candidatas))
print("Excluidas por más de 95% de faltantes:", len(columnas_mas95))
print("Predictoras finales antes de codificar:", len(columnas_modelo))
assert len(columnas_modelo) == 65, f"Se esperaban 65 predictoras; se encontraron {len(columnas_modelo)}. Revisar cambios del archivo o reglas."

X_train_raw = train_raw[columnas_modelo]
X_valid_raw = valid_raw[columnas_modelo]
y_train = train_df["default"].to_numpy()
y_valid = valid_df["default"].to_numpy()
preprocesador, _ = construir_preprocesador(X_train_raw, con_indicador=[])
preprocesador.fit(X_train_raw)
nombres_codificados = preprocesador.get_feature_names_out().tolist()
X_train = preprocesador.transform(X_train_raw)
X_valid = preprocesador.transform(X_valid_raw)
print("Columnas después de codificar categorías:", len(nombres_codificados))
print("Matrices de entrenamiento y validación:", X_train.shape, X_valid.shape)

# Relaciona cada columna creada por One-Hot Encoding con su variable original.
fuente_codificada = {}
for nombre in nombres_codificados:
    posibles = [c for c in columnas_modelo if nombre == c or nombre.startswith(c + "_")]
    if not posibles:
        raise ValueError(f"No se pudo relacionar {nombre} con la variable original.")
    fuente_codificada[nombre] = max(posibles, key=len)
indices_por_variable = {
    c: [i for i, nombre in enumerate(nombres_codificados) if fuente_codificada[nombre] == c]
    for c in columnas_modelo
}
print("Variables con columnas codificadas:", sum(bool(v) for v in indices_por_variable.values()))

Predictoras originales antes de revisar faltantes: 95
Excluidas por más de 95% de faltantes: 30
Predictoras finales antes de codificar: 65


C:\Users\josue\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Columnas después de codificar categorías: 139
Matrices de entrenamiento y validación: (451059, 139) (668640, 139)
Variables con columnas codificadas: 64


## 3. Referencia: regresión logística con las 65 variables

Entrenamos la regresión logística actual para comparar el rendimiento del conjunto completo con las versiones reducidas. La métrica principal del proyecto es ROC-AUC; también reportamos PR-AUC por el desbalance entre clases.

In [4]:
modelo_65 = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
modelo_65.fit(X_train, y_train)
score_valid_65 = modelo_65.predict_proba(X_valid)[:, 1]
resultado_65 = {
    "variables originales": len(columnas_modelo),
    "columnas codificadas": X_train.shape[1],
    "ROC-AUC validación": roc_auc_score(y_valid, score_valid_65),
    "PR-AUC validación": average_precision_score(y_valid, score_valid_65),
}
display(pd.Series(resultado_65).to_frame("baseline con 65 variables"))

,baseline con 65 variables
variables originales,65.000000
columnas codificadas,139.000000
ROC-AUC validación,0.715589
PR-AUC validación,0.409262


## 4. Método 1: regresión logística L1

L1 puede llevar algunos coeficientes exactamente a cero. Como una variable categórica genera varias columnas, contamos tanto las columnas activas como las variables originales que conservan al menos un coeficiente distinto de cero. Usamos una muestra reproducible del entrenamiento para que la selección sea práctica; no tocamos validación ni prueba para decidir qué columnas retener.

In [5]:
rng = np.random.default_rng(SEMILLA)
tamano_seleccion = min(100_000, len(y_train))
indices_seleccion = rng.choice(len(y_train), size=tamano_seleccion, replace=False)
X_seleccion = X_train[indices_seleccion]
y_seleccion = y_train[indices_seleccion]
modelo_l1 = LogisticRegression(
    C=0.1, l1_ratio=1.0, solver="saga", max_iter=300, tol=0.002,
    random_state=SEMILLA
)
with warnings.catch_warnings(record=True) as avisos_l1:
    warnings.simplefilter("always")
    modelo_l1.fit(X_seleccion, y_seleccion)
from sklearn.exceptions import ConvergenceWarning
print("Iteraciones de L1:", int(modelo_l1.n_iter_[0]))
avisos_convergencia = [str(a.message) for a in avisos_l1 if issubclass(a.category, ConvergenceWarning)]
if avisos_convergencia:
    print("L1 no convergió con el máximo de iteraciones:", avisos_convergencia[0])
coef_l1 = np.abs(modelo_l1.coef_[0])
activas_l1 = np.flatnonzero(coef_l1 > 1e-6)
variables_l1 = sorted({fuente_codificada[nombres_codificados[i]] for i in activas_l1})
print("Filas usadas para ajustar L1:", len(y_seleccion))
print("Columnas codificadas con coeficiente no nulo:", len(activas_l1), "de", len(nombres_codificados))
print("Variables originales conservadas por L1:", len(variables_l1), "de", len(columnas_modelo))

display(pd.DataFrame({"variable_original": variables_l1,
                      "nombre_espanol": [nombres_variables.get(v, v) for v in variables_l1]}))

Iteraciones de L1: 111
Filas usadas para ajustar L1: 100000
Columnas codificadas con coeficiente no nulo: 121 de 139
Variables originales conservadas por L1: 60 de 65


,variable_original,nombre_espanol
0,acc_now_delinq,cuentas_en_mora_actual
1,acc_open_past_24mths,cuentas_abiertas_ultimos_24m
2,addr_state,estado_residencia
3,annual_inc,ingreso_anual
4,avg_cur_bal,saldo_promedio_cuentas
5,bc_open_to_buy,credito_disponible_tarjetas
6,bc_util,utilizacion_tarjetas_pct
7,chargeoff_within_12_mths,cuentas_castigadas_12m
8,collections_12_mths_ex_med,cobranzas_no_medicas_12m
9,delinq_2yrs,moras_ultimos_2_anios


## 5. Método 2: importancia por permutación agrupada

Se mide cuánto baja ROC-AUC al mezclar los valores de una variable original. Esto agrupa en una sola medida todas las columnas One-Hot de esa variable. Para no usar la validación final durante la selección, entrenamos este modelo con préstamos anteriores a 2014 y calculamos importancia en una muestra de 2014. Presentamos las 30 primeras como un escenario de reducción a 30; es una decisión de tamaño que luego comparamos en la validación 2015–2016.

In [6]:
anio_train = train_df["anio"]
mascara_interno_train = anio_train < 2014
mascara_importancia = anio_train == 2014
X_interno = X_train_raw.loc[mascara_interno_train]
y_interno = y_train[mascara_interno_train.to_numpy()]
X_importancia = X_train_raw.loc[mascara_importancia]
y_importancia = y_train[mascara_importancia.to_numpy()]
if len(X_importancia) == 0 or len(X_interno) == 0:
    raise ValueError("No hay filas suficientes antes de 2014 y en 2014 para la permutación.")

# Limitar filas de cálculo, manteniendo el bloque temporal y la semilla fija.
max_interno = min(120_000, len(X_interno))
max_importancia = min(20_000, len(X_importancia))
idx_interno = rng.choice(len(X_interno), max_interno, replace=False)
idx_importancia = rng.choice(len(X_importancia), max_importancia, replace=False)
X_interno_muestra = X_interno.iloc[idx_interno]
y_interno_muestra = y_interno[idx_interno]
X_importancia_muestra = X_importancia.iloc[idx_importancia]
y_importancia_muestra = y_importancia[idx_importancia]

pre_perm, _ = construir_preprocesador(X_interno_muestra, con_indicador=[])
X_interno_cod = pre_perm.fit_transform(X_interno_muestra)
modelo_perm = LogisticRegression(C=1.0, max_iter=1500, solver="lbfgs", random_state=SEMILLA)
modelo_perm.fit(X_interno_cod, y_interno_muestra)
pipeline_perm = Pipeline([("preprocesador", pre_perm), ("modelo", modelo_perm)])
resultado_perm = permutation_importance(
    pipeline_perm, X_importancia_muestra, y_importancia_muestra,
    scoring="roc_auc", n_repeats=2, random_state=SEMILLA, n_jobs=1
)
tabla_perm = pd.DataFrame({
    "variable": X_importancia_muestra.columns,
    "importancia_media": resultado_perm.importances_mean,
    "variacion": resultado_perm.importances_std,
}).sort_values("importancia_media", ascending=False).reset_index(drop=True)
variables_perm30 = tabla_perm.head(30)["variable"].tolist()
print(f"Filas del bloque interno de entrenamiento: {len(y_interno_muestra):,}")
print(f"Filas usadas para importancia en 2014: {len(y_importancia_muestra):,}")
print("Variables en la lista top 30:", len(variables_perm30))
tabla_perm_presentacion = tabla_perm.head(30).copy()
tabla_perm_presentacion["variable"] = tabla_perm_presentacion["variable"].map(lambda v: nombres_variables.get(v, v))
display(tabla_perm_presentacion)

Filas del bloque interno de entrenamiento: 120,000
Filas usadas para importancia en 2014: 20,000
Variables en la lista top 30: 30


,variable,importancia_media,variacion
0,plazo_meses,0.059807,0.002358
1,monto_prestamo,0.017255,0.000671
2,revolventes_con_saldo,0.014190,0.000024
3,ingreso_anual,0.012704,0.000369
4,limite_total_tarjetas,0.010518,0.000329
5,cuentas_abiertas_ultimos_24m,0.009490,0.000499
6,ratio_deuda_ingreso,0.007770,0.000215
7,limite_credito_total,0.007120,0.000074
8,proposito_prestamo,0.006109,0.001156
9,saldo_total_actual,0.005362,0.000403


## 6. Método 3: eliminación recursiva agrupada (RFE)

Eliminamos grupos de columnas codificadas hasta conservar 30 variables originales. En cada ronda entrenamos una logística y medimos la fuerza de cada variable con la norma de sus coeficientes (todas sus categorías juntas). El proceso usa solo la muestra de entrenamiento; por eso el resultado es una selección candidata, no una prueba de que 30 sea el número óptimo.

In [7]:
objetivo_rfe = min(30, len(columnas_modelo))
activas_rfe = list(columnas_modelo)
historial_rfe = []
X_rfe = X_seleccion

while len(activas_rfe) > objetivo_rfe:
    indices_activos = sorted(i for v in activas_rfe for i in indices_por_variable[v])
    modelo_rfe = LogisticRegression(C=1.0, max_iter=800, solver="lbfgs", random_state=SEMILLA)
    modelo_rfe.fit(X_rfe[:, indices_activos], y_seleccion)
    coef = modelo_rfe.coef_[0]
    fuerza = {}
    for variable in activas_rfe:
        indices_locales = [indices_activos.index(i) for i in indices_por_variable[variable]]
        fuerza[variable] = float(np.linalg.norm(coef[indices_locales]))
    quitar_n = min(5, len(activas_rfe) - objetivo_rfe)
    quitar = sorted(activas_rfe, key=lambda v: fuerza[v])[:quitar_n]
    historial_rfe.append({"variables_antes": len(activas_rfe), "retiradas": quitar})
    activas_rfe = [v for v in activas_rfe if v not in quitar]
variables_rfe30 = activas_rfe
print("Rondas RFE:", len(historial_rfe))
print("Variables originales conservadas por RFE agrupado:", len(variables_rfe30))
display(pd.DataFrame({"variable_original": variables_rfe30,
                      "nombre_espanol": [nombres_variables.get(v, v) for v in variables_rfe30]}))

Rondas RFE: 7
Variables originales conservadas por RFE agrupado: 30


,variable_original,nombre_espanol
0,loan_amnt,monto_prestamo
1,term,plazo_meses
2,emp_length,antiguedad_laboral
3,home_ownership,tenencia_vivienda
4,annual_inc,ingreso_anual
5,verification_status,verificacion_ingresos
6,purpose,proposito_prestamo
7,addr_state,estado_residencia
8,dti,ratio_deuda_ingreso
9,delinq_2yrs,moras_ultimos_2_anios


## 7. Método 4: PCA

PCA combina columnas en componentes nuevos. Por eso reduce el número de dimensiones del modelo, pero no permite decir que seleccionó cierto número de las 65 variables originales. Ajustamos PCA a una muestra del entrenamiento y buscamos conservar al menos 95% de la variación observada.

In [8]:
tamano_pca = min(80_000, len(X_train))
idx_pca = rng.choice(len(X_train), size=tamano_pca, replace=False)
pca = PCA(n_components=0.95, svd_solver="full", random_state=SEMILLA)
pca.fit(X_train[idx_pca])
print("Variables originales de entrada:", len(columnas_modelo))
print("Columnas codificadas de entrada:", X_train.shape[1])
print("Componentes PCA para conservar 95% de variación:", pca.n_components_)
print("Porcentaje acumulado explicado:", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

Variables originales de entrada: 65
Columnas codificadas de entrada: 139
Componentes PCA para conservar 95% de variación: 46
Porcentaje acumulado explicado: 95.16 %


## 8. Comparar los conjuntos reducidos en validación

Ahora comparamos 65 variables con las selecciones L1, permutación top 30 y RFE 30. PCA se reporta por separado como número de componentes. Todos los modelos de comparación se ajustan con entrenamiento y se miden en 2015–2016. La prueba desde 2017 sigue sin consultarse.

In [9]:
def indices_de_variables(variables):
    return sorted(i for v in variables for i in indices_por_variable[v])

def evaluar_subconjunto(nombre, variables):
    indices = indices_de_variables(variables)
    modelo = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
    modelo.fit(X_train[:, indices], y_train)
    score = modelo.predict_proba(X_valid[:, indices])[:, 1]
    return {
        "metodo": nombre,
        "variables_originales": len(variables),
        "columnas_codificadas": len(indices),
        "ROC-AUC_validacion": roc_auc_score(y_valid, score),
        "PR-AUC_validacion": average_precision_score(y_valid, score),
    }

filas_comparacion = [{
    "metodo": "Baseline 65",
    "variables_originales": len(columnas_modelo),
    "columnas_codificadas": X_train.shape[1],
    "ROC-AUC_validacion": roc_auc_score(y_valid, score_valid_65),
    "PR-AUC_validacion": average_precision_score(y_valid, score_valid_65),
}]
for nombre, variables in [
    ("L1 agrupada", variables_l1),
    ("Permutación top 30", variables_perm30),
    ("RFE agrupado 30", variables_rfe30),
]:
    filas_comparacion.append(evaluar_subconjunto(nombre, variables))

X_train_pca = pca.transform(X_train)
X_valid_pca = pca.transform(X_valid)
modelo_pca = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
modelo_pca.fit(X_train_pca, y_train)
score_pca = modelo_pca.predict_proba(X_valid_pca)[:, 1]
filas_comparacion.append({
    "metodo": "PCA 95% (componentes)",
    "variables_originales": len(columnas_modelo),
    "columnas_codificadas": int(pca.n_components_),
    "ROC-AUC_validacion": roc_auc_score(y_valid, score_pca),
    "PR-AUC_validacion": average_precision_score(y_valid, score_pca),
})
comparacion = pd.DataFrame(filas_comparacion)
with pd.option_context("display.max_columns", None, "display.width", 140):
    display(comparacion[["metodo", "variables_originales", "columnas_codificadas", "ROC-AUC_validacion", "PR-AUC_validacion"]].round(4))

,metodo,variables_originales,columnas_codificadas,ROC-AUC_validacion,PR-AUC_validacion
0,Baseline 65,65,139,0.7156,0.4093
1,L1 agrupada,60,135,0.7156,0.4093
2,Permutación top 30,30,104,0.7134,0.4072
3,RFE agrupado 30,30,105,0.7149,0.4086
4,PCA 95% (componentes),65,46,0.7072,0.3999


## 9. Conclusión y decisión provisional

La cantidad final depende del método: L1 decide según los coeficientes; permutación y RFE se fijan en 30 para responder al escenario propuesto; PCA devuelve componentes, no variables originales. Para recomendar la reducción de 65 a 30, compara las métricas de validación con el baseline. Si el rendimiento baja de forma apreciable, conviene conservar más variables o revisar el tratamiento. No se usa el conjunto de prueba para elegir.

In [10]:
resumen_metodos = pd.DataFrame([
    {"método": "Baseline", "variables_originales": len(columnas_modelo), "resultado": "referencia"},
    {"método": "L1", "variables_originales": len(variables_l1), "resultado": "selección por coeficientes no nulos"},
    {"método": "Permutación agrupada", "variables_originales": len(variables_perm30), "resultado": "top 30 por importancia"},
    {"método": "RFE agrupado", "variables_originales": len(variables_rfe30), "resultado": "30 conservadas"},
    {"método": "PCA", "variables_originales": len(columnas_modelo), "resultado": f"{pca.n_components_} componentes, no selecciona columnas originales"},
])
display(resumen_metodos)
print("La selección candidata de 30 variables originales queda definida por el RFE agrupado.")
print("Su resultado de validación está en la tabla comparativa; la prueba permanece reservada.")

,método,variables_originales,resultado
0,Baseline,65,referencia
1,L1,60,selección por coeficientes no nulos
2,Permutación agrupada,30,top 30 por importancia
3,RFE agrupado,30,30 conservadas
4,PCA,65,"46 componentes, no selecciona columnas originales"


La selección candidata de 30 variables originales queda definida por el RFE agrupado.
Su resultado de validación está en la tabla comparativa; la prueba permanece reservada.
